# Week 3 — BANKING77 seed confirmation (E1) + rank sweep (E3) (Kunj)
Uses the learning rates picked in Week 2 (`results/selected_lr_banking77.json`).

- **A. Confirmation:** head_only, lora_r8, full_ft at seeds 43 and 44 (6 runs)
- **B. E3 rank sweep:** LoRA r4 and r16, seed 42, same LR as LoRA r8 (2 runs)
- **C. Supplementary (optional):** head-only at 1e-3 and 3e-3, seed 42 (2 runs). Not part of the pre-registered grid; reported separately. Set `RUN_SUPPLEMENTARY = False` to skip.

**Kaggle settings:** Accelerator = **GPU T4 x2**, Internet = **On**. Run with **Save Version → Save & Run All (Commit)**. About 40–50 min.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # one T4, matching the proposal
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone https://github.com/devjp99/lora-on-a-budget.git
%cd lora-on-a-budget
!git checkout kunj/week3-confirm-e3
!git log --oneline -3

In [ ]:
!pip uninstall -y -q torchao   # Kaggle ships torchao 0.10, which peft rejects; we don't use it
!pip install -q -r requirements.txt
!pip install -q -e .
os.environ['PYTHONPATH'] = 'src'

## 1. Rebuild manifests and run tests

In [ ]:
!python scripts/make_banking77_manifest.py
!git status --short manifests/   # empty output = identical split
!LORA_BUDGET_ONLINE=1 python -m pytest -q

## 2. Build the job list from the Week 2 picks

In [ ]:
import json
RUN_SUPPLEMENTARY = True
CKPT = '/kaggle/working/checkpoints' if os.path.isdir('/kaggle/working') else 'checkpoints'
sel = json.load(open('results/selected_lr_banking77.json'))
jobs = []
for name in ['head_only', 'lora_r8', 'full_ft']:          # A. confirmation
    for seed in [43, 44]:
        jobs.append((name, sel[name]['lr'], seed, 'confirm'))
for name in ['lora_r4', 'lora_r16']:                     # B. E3, reuse the E1 LoRA LR
    jobs.append((name, sel['lora_r8']['lr'], 42, 'rank_sweep'))
if RUN_SUPPLEMENTARY:                                    # C. extra head-only check
    for lr in [1e-3, 3e-3]:
        jobs.append(('head_only', lr, 42, 'supplementary'))
for j in jobs:
    print(j)
print(len(jobs), 'jobs')

## 3. Run them

In [ ]:
import time
for name, lr, seed, phase in jobs:
    print(f'\n===== {phase}  {name}  lr={lr}  seed={seed}  ({time.strftime("%H:%M:%S")}) =====')
    !python scripts/train_banking77.py --config configs/banking77/{name}.yaml --lr {lr} --seed {seed} --phase {phase} --ckpt-root {CKPT}

## 4. Summary (tuning set only; the test set is still untouched)

In [ ]:
import pandas as pd
led = pd.read_csv('results/train_ledger.csv').drop_duplicates('run_id', keep='last')
print(led['status'].value_counts().to_string())
# 3-seed view: the chosen seed-42 run + the two confirmation runs
picked = [sel[n]['seed42_run_id'] for n in ['head_only', 'lora_r8', 'full_ft']]
three = led[led['run_id'].isin(picked) | (led['phase'] == 'confirm')]
print('\nE1, 3 seeds (tuning set):')
print(three.groupby('method')[['tune_accuracy', 'tune_macro_f1', 'peak_alloc_mb', 'train_s']].agg(['mean', 'std']).round(2).to_string())
r8 = led[led['run_id'] == sel['lora_r8']['seed42_run_id']]
ranks = pd.concat([led[led['phase'] == 'rank_sweep'], r8])
print('\nE3, seed 42 (tuning set):')
print(ranks[['rank', 'tune_accuracy', 'tune_macro_f1', 'trainable_params', 'peak_alloc_mb', 'train_s', 'checkpoint_mb']].sort_values('rank').to_string(index=False))
sup = led[led['phase'] == 'supplementary']
if len(sup):
    print('\nSupplementary head-only (seed 42, tuning set):')
    print(sup[['lr', 'tune_accuracy', 'tune_macro_f1', 'best_epoch']].to_string(index=False))

## 5. Save results for download

In [ ]:
import shutil
out_dir = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
shutil.make_archive(os.path.join(out_dir, 'week3_results'), 'zip', '.', 'results')
print('saved', os.path.join(out_dir, 'week3_results.zip'))
!du -sh {CKPT}/* 2>/dev/null